# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedman47/ML-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Task type: ranking (scored by a classifier). Each page gets a risk score, and the review queue is the pages sorted by that score. The decision is "which pages do I review first?", so the order matters, not a yes/no per page.

Why ranking and not plain classification:a reviewer has limited capacity (about 20–50 pages), so only the top of the list is ever used. Clustering doesn't fit because I already know what I want to find (declining pages).

In [1]:
import os, subprocess
import numpy as np, pandas as pd

# move to repo root (Colab or local)
if "google.colab" in str(get_ipython()):
    if not os.path.isdir("data/raw"):
        if not os.path.isdir("flyrank-ml-internship-starter"):
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
        os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# unit of analysis (same filters as the reference pipeline)
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)] \
        .drop_duplicates("content_id").copy()
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# a first hand-rule score, used here only to show the OUTPUT SHAPE of the queue
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["rule_score"] = stale * visible * df["impressions_90d"]

queue = (df.sort_values("rule_score", ascending=False).head(5)
           [["content_id", "rule_score", "impressions_90d", "days_since_last_update"]].copy())
queue["reason_code"] = "stale_visible_page"
queue["suggested_action"] = "review for refresh"
print("Sketch of the output I want: a ranked queue with reason codes")
display(queue)

Sketch of the output I want: a ranked queue with reason codes


,content_id,rule_score,impressions_90d,days_since_last_update,reason_code,suggested_action
16751,content_cf56e2e2e282,61678,61678,194,stale_visible_page,review for refresh
16514,content_7368877ea310,59472,59472,194,stale_visible_page,review for refresh
7021,content_1bfaa38ff26c,25715,25715,194,stale_visible_page,review for refresh
21268,content_0a91db491d14,13299,13299,193,stale_visible_page,review for refresh
11489,content_5feee3994adb,7812,7812,194,stale_visible_page,review for refresh


**Target (proxy label):** `is_declining = 1` when `trend_direction == "down"`, i.e. impressions in the last 30 days fell by more than 20% versus the previous 30 days.

**Where it comes from:** a *defined rule* applied to observed impressions, not a future outcome. That makes it a **proxy**: it measures a drop that already happened inside the current window, and it could be noise, seasonality or consolidation.

**Excluded from features:** `trend_direction` and `trend_pct` (the label is computed from them), plus `content_id` and `client_id` (grouping only).

**Planned upgrade:** in Weeks 3–4, a future-window label (features from the prior window, decline measured in the next 30 days).

In [2]:
print("Label balance:", df["is_declining"].value_counts(normalize=True).round(3).to_dict())
print("trend_direction:", df["trend_direction"].value_counts().to_dict())

# the label really is the 20% rule on observed windows
pct = (df["impressions_last_30d"] - df["impressions_prev_30d"]) \
      / df["impressions_prev_30d"].replace(0, np.nan) * 100
down = df["trend_direction"] == "down"
print(f"'down' pages with a real drop of more than 20%: {(down & (pct < -20)).sum():,} of {down.sum():,}")

label_source = ["trend_direction", "trend_pct"]
id_cols = ["content_id", "client_id"]
candidate_features = [c for c in df.columns if c not in label_source + id_cols + ["is_declining", "rule_score"]]
print(f"{len(candidate_features)} candidate feature columns; label-source and ID columns excluded")

Label balance: {1: 0.542, 0: 0.458}
trend_direction: {'down': 16262, 'stable': 5962, 'up': 4388, 'new': 2236, 'flat': 1152}
'down' pages with a real drop of more than 20%: 16,262 of 16,262
40 candidate feature columns; label-source and ID columns excluded


**Metric: Precision@K (K = 20 and 50), reported as lift over the base rate.**

Precision@K asks: of the K pages at the top of my queue, what fraction truly declined? K matches reviewer capacity.

**What "good" means:** about 54% of all pages are labelled declining, so a random queue already scores about 0.54. A useful model must clearly beat that, and must beat the transparent hand rule, on **clients it was not trained on**. Average precision is a secondary check on the whole ranking.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining"].values
base = y.mean()
print(f"Base rate (random queue): {base:.3f}")
for k in (20, 50, 200):
    p = precision_at_k(df["rule_score"], y, k)
    print(f"Hand rule  Precision@{k}: {p:.3f}   (lift over random: {p/base:.2f}x)")

Base rate (random queue): 0.542
Hand rule  Precision@20: 0.950   (lift over random: 1.75x)
Hand rule  Precision@50: 0.740   (lift over random: 1.37x)
Hand rule  Precision@200: 0.605   (lift over random: 1.12x)


**One row = one page (content item)** in the last 90 days, with its search, engagement and content measurements. There are about 30,000 pages across 32 anonymized clients. Pages from the same client share patterns, so I will validate by holding out whole clients.

In [4]:
cols = ["content_id", "client_id", "content_type", "content_age_days",
        "days_since_last_update", "impressions_90d", "avg_position", "ctr",
        "trend_direction", "is_declining"]
print(f"{len(df):,} rows (pages) x {df['client_id'].nunique()} clients")
display(df[cols].head(8))

print("\nSketch of my target column:")
display(df[["content_id", "impressions_last_30d", "impressions_prev_30d",
            "trend_direction", "is_declining"]].head(5))

30,000 rows (pages) x 32 clients


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction,is_declining
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,3803,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,15320,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,12581,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,11751,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,19140,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,3970,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,1724,21.2,0.06,stable,0



Sketch of my target column:


,content_id,impressions_last_30d,impressions_prev_30d,trend_direction,is_declining
0,content_304f48230142,578,987,down,1
1,content_a1fb4e703a9e,2501,5915,down,1
2,content_9aa793d4d895,2382,6089,down,1
3,content_331d6c4de07b,3626,4206,stable,0
4,content_d99b7a2d90ca,4211,6452,down,1


**The case for ML:** a hand rule uses one or two signals and fixed cutoffs (stale ≥ 180 days AND ≥ 500 impressions). Declines probably depend on many signals at once (position, age, freshness, engagement, content type) and on interactions between them, which are hard to write as if-statements.

**What I observed:** the hand rule scores zero for all but a handful of pages, so beyond its first few hits the top-50 is mostly ties (arbitrary order), and its Precision@K is partly luck. That is a real limit of a fixed rule: it ranks very few pages.

**What I have not shown yet:** whether a learned model beats it on unseen clients. The comparison below is a first check, and a model only earns its place if it wins there.

In [5]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

print(f"Pages the hand rule gives a non-zero score: {(df['rule_score'] > 0).sum()} of {len(df):,}")

feats = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position",
         "ctr", "word_count", "sessions_90d", "engagement_rate", "search_volume"]
X = df[feats].replace([np.inf, -np.inf], np.nan).fillna(0)

# hold out whole clients (seed 42)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
              .split(X, y, groups=df["client_id"]))
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced",
                            random_state=42, n_jobs=-1).fit(X.iloc[tr], y[tr])
score = rf.predict_proba(X.iloc[te])[:, 1]
y_te, rule_te = y[te], df["rule_score"].values[te]

print(f"Held-out: {len(te):,} pages from {df['client_id'].iloc[te].nunique()} unseen clients, "
      f"base rate {y_te.mean():.3f}")
for k in (20, 50, 200):
    print(f"Precision@{k}:  hand rule {precision_at_k(rule_te, y_te, k):.3f}   "
          f"random forest {precision_at_k(score, y_te, k):.3f}")

Pages the hand rule gives a non-zero score: 17 of 30,000
Held-out: 10,834 pages from 10 unseen clients, base rate 0.559
Precision@20:  hand rule 0.750   random forest 0.550
Precision@50:  hand rule 0.720   random forest 0.660
Precision@200:  hand rule 0.610   random forest 0.620


## Self-check

Before you submit, confirm each line honestly:

- [yes] Every section above is filled — markdown thinking AND the code that backs it
- [yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [yes] No client names, URLs, or private queries anywhere
- [yes] My claims use careful words: observed, measured, directional, decision-support
- [yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.